In [ ]:
!git clone https://github.com/Exploration-Lab/CS780-OBELIX.git
%cd CS780-OBELIX
!pip install -r requirements.txt

Cloning into 'CS780-OBELIX'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 32 (delta 6), reused 4 (delta 4), pack-reused 24 (from 1)
Receiving objects: 100% (32/32), 993.89 KiB | 31.06 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/CS780-OBELIX/CS780-OBELIX


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_PATH = "/content/drive/MyDrive/obelix_dqrn.pth"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


ImportError: cannot import name 'ObelixEnv' from 'obelix' (/content/CS780-OBELIX/obelix.py)

In [ ]:
class DQRN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(18, 128)
        self.lstm = nn.LSTM(128, 128, batch_first=True)
        self.fc2 = nn.Linear(128, 5)

    def forward(self, x, hidden):
        x = F.relu(self.fc1(x))
        out, hidden = self.lstm(x, hidden)
        return self.fc2(out), hidden

    def init_hidden(self, batch_size=1):
        return (torch.zeros(1, batch_size, 128),
                torch.zeros(1, batch_size, 128))

In [ ]:
def shape_reward(obs, reward):
    # obs = 18 bits

    forward_near = obs[2]   # adjust index if needed
    forward_far = obs[3]
    infrared = obs[16]
    attached = obs[17]

    shaped = reward

    # Encourage moving toward box
    if forward_near:
        shaped += 2
    if forward_far:
        shaped += 1

    # Strong signal when aligned
    if infrared:
        shaped += 3

    # BIG reward for pushing state
    if attached:
        shaped += 5

    return shaped

In [ ]:
model = DQRN()
target_model = DQRN()

if os.path.exists(SAVE_PATH):
    print("🔁 Loading saved model...")
    model.load_state_dict(torch.load(SAVE_PATH, map_location="cpu"))

target_model.load_state_dict(model.state_dict())

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
buffer = ReplayBuffer()

In [ ]:
episodes = 1000
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995

for ep in range(episodes):
    obs, _ = env.reset()
    done = False

    agent = Agent(model)
    episode_data = []
    total_reward = 0

    while not done:
        action = agent.act(obs, epsilon)

        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # 🔥 APPLY REWARD SHAPING
        reward = shape_reward(next_obs, reward)

        episode_data.append((obs, action, reward, next_obs, done))

        obs = next_obs
        total_reward += reward

    buffer.push(episode_data)

    train_step(model, target_model, buffer, optimizer)

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    # Update target network
    if ep % 10 == 0:
        target_model.load_state_dict(model.state_dict())

    # 🔥 SAVE CHECKPOINT
    if ep % 20 == 0:
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"💾 Saved model at episode {ep}")

    print(f"Episode {ep} | Reward: {total_reward:.1f} | Eps: {epsilon:.3f}")

In [ ]:
class Agent:
    def __init__(self, model):
        self.model = model
        self.hidden = model.init_hidden(1)

    def reset(self):
        self.hidden = self.model.init_hidden(1)

    def act(self, obs, epsilon):
        if random.random() < epsilon:
            return random.randint(0, 4)

        obs = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        q, self.hidden = self.model(obs, self.hidden)
        return q.argmax(dim=-1).item()

In [ ]:
class Callback:
    def __init__(self, freq=20):
        self.freq = freq
        self.best = -1e9

    def __call__(self, agent, ep, reward):

        if ep % self.freq == 0:
            print("💾 Autosaving...")
            agent.save()

        if reward > self.best:
            self.best = reward
            torch.save(agent.q.state_dict(), WEIGHTS_PATH)
            print("🔥 Best model saved:", reward)

In [ ]:
def train(episodes=300):

    env = ObelixEnv()
    agent = Agent()

    agent.load()

    cb = Callback()

    for ep in range(episodes):

        state,_ = env.reset()
        total = 0

        hidden = agent.q.init_hidden(1)

        for t in range(500):

            action, hidden = agent.act(state, hidden)

            # take step
            next_state, reward, done, _, _ = env.step(action)

            # 🔥 reward shaping: encourage forward movement
            if action == 2:  # 2 = "FW"
                reward += 1.0

            # 🔥 encourage seeing the box (VERY important)
            if sum(next_state) > 0:
                reward += 0.5

            agent.buffer.add((state,action,reward,next_state,done))

            # 🔥 TRAIN (delayed + less frequent)
            if agent.steps % 4 == 0:
                agent.train()

            agent.steps += 1

            # 🔥 EPSILON DECAY (PUT IT HERE)
            agent.eps = max(agent.eps - 4e-6, agent.eps_min)

            # target update
            if agent.steps % 1000 == 0:
                agent.target.load_state_dict(agent.q.state_dict())

            state = next_state
            total += reward

            if done:
                break

        print(f"Ep {ep} | Reward {total} | Eps {agent.eps:.3f} | Buffer {len(agent.buffer.buffer)}")

        cb(agent, ep, total)

    return agent

In [ ]:
agent = train(episodes=800)

⚠️ No checkpoint found
Ep 0 | Reward -398.0 | Eps 0.998 | Buffer 500
💾 Autosaving...
🔥 Best model saved: -398.0
Ep 1 | Reward -250.5 | Eps 0.996 | Buffer 1000
🔥 Best model saved: -250.5
Ep 2 | Reward -417.5 | Eps 0.994 | Buffer 1500
Ep 3 | Reward -403.0 | Eps 0.992 | Buffer 2000
Ep 4 | Reward -305.5 | Eps 0.990 | Buffer 2500
Ep 5 | Reward -407.0 | Eps 0.988 | Buffer 3000
Ep 6 | Reward -255.5 | Eps 0.986 | Buffer 3500
Ep 7 | Reward -386.0 | Eps 0.984 | Buffer 4000
Ep 8 | Reward -267.5 | Eps 0.982 | Buffer 4500
Ep 9 | Reward -439.0 | Eps 0.980 | Buffer 5000
Ep 10 | Reward -380.0 | Eps 0.978 | Buffer 5500
Ep 11 | Reward -207.5 | Eps 0.976 | Buffer 6000
🔥 Best model saved: -207.5
Ep 12 | Reward -395.5 | Eps 0.974 | Buffer 6500
Ep 13 | Reward -372.0 | Eps 0.972 | Buffer 7000
Ep 14 | Reward -404.0 | Eps 0.970 | Buffer 7500
Ep 15 | Reward -409.0 | Eps 0.968 | Buffer 8000
Ep 16 | Reward -401.0 | Eps 0.966 | Buffer 8500
Ep 17 | Reward -359.5 | Eps 0.964 | Buffer 9000
Ep 18 | Reward -411.0 | Eps

KeyboardInterrupt: 